In [ ]:
BUENOS_AIRES = "BUENOS_AIRES"
FERNANDEZ = "FERNANDEZ"

DISPONIBLE = 0
CARGANDO = 1
VIAJANDO = 2
DESCARGANDO = 3
CARGANDO_COMBUSTIBLE = 4

ESCALA_TIEMPO = 0.01

In [ ]:
import threading
import time
import random

class Camion:
    def __init__(self, numero):
        self.numero = numero
        self.ubicacion = "BUENOS_AIRES"
        self.estado = "DISPONIBLE"


class Recursos:
    def __init__(self, x):
        self.carga_buenos_aires = threading.Semaphore(1)
        self.carga_fernandez = threading.Semaphore(1)

        self.descarga_buenos_aires = threading.Semaphore(1)
        self.descarga_fernandez = threading.Semaphore(1)

        self.surtidores = threading.Semaphore(2)

        self.viajes_buenos_aires = x
        self.viajes_fernandez = x

        self.lock_viajes = threading.Lock()

In [ ]:
def viajar(camion, destino):
    camion.estado = VIAJANDO
    print(f"Camión {camion.numero}: VIAJANDO")

    horas = random.uniform(18, 24)
    time.sleep(horas * ESCALA_TIEMPO)

    camion.ubicacion = destino

def cargar(camion, planta, recurso):
    camion.estado = CARGANDO
    print(f"Camión {camion.numero}: CARGANDO EN {planta}")

    with recurso:                         # With es la estructura que simplifica tener que usar acquire() y release() con un atributo semaforo.
        time.sleep(2 * ESCALA_TIEMPO)


def descargar(camion, planta, recurso):
    camion.estado = DESCARGANDO
    print(f"Camión {camion.numero}: DESCARGANDO EN {planta}")

    with recurso:
        time.sleep(2 * ESCALA_TIEMPO)


def cargar_combustible(camion, recursos):
    camion.estado = CARGANDO_COMBUSTIBLE
    print(f"Camión {camion.numero}: CARGANDO COMBUSTIBLE")

    with recursos.surtidores:
        time.sleep(1 * ESCALA_TIEMPO)

Para consultar o modificar viajes_buenos_aires, todos los hilos tienen que adquirir primero lock_viajes. Es decir, todos los hilos tienen acceso al mismo objeto Recursos, pero eso no quiere decir que tienen acceso al atributo, o más bien, al bloque de codigo especifico. Primero van a tener que pasar por el atributo lock para poder modificar los atributos semaforos.

El Lock no tiene conocimiento de si estás leyendo o modificando. Simplemente bloquea a otros hilos que intenten adquirir ese mismo Lock.

In [ ]:
def trabajar(camion, recursos):
    while True:

        if camion.ubicacion == BUENOS_AIRES:

            with recursos.lock_viajes:
                if recursos.viajes_buenos_aires == 0:
                    break                               # El break sale de bucles FOR/WHILE

                recursos.viajes_buenos_aires -= 1

            cargar(
                camion,
                BUENOS_AIRES,
                recursos.carga_buenos_aires
            )

            viajar(camion, FERNANDEZ)

            descargar(
                camion,
                FERNANDEZ,
                recursos.descarga_fernandez
            )

        elif camion.ubicacion == FERNANDEZ:

            with recursos.lock_viajes:
                if recursos.viajes_fernandez == 0:
                    break

                recursos.viajes_fernandez -= 1

            cargar(
                camion,
                FERNANDEZ,
                recursos.carga_fernandez
            )

            cargar_combustible(camion, recursos)

            viajar(camion, BUENOS_AIRES)

            descargar(
                camion,
                BUENOS_AIRES,
                recursos.descarga_buenos_aires
            )

In [ ]:
def iniciar_simulacion(n, x):
    recursos = Recursos(x)
    inicio = time.time()
    hilos = []

    for i in range(n):
        camion = Camion(i + 1)

        hilo = threading.Thread(
            target=trabajar,
            args=(camion, recursos)
        )

        hilos.append(hilo)
        hilo.start()

    for hilo in hilos:
        hilo.join()

    fin = time.time()

    tiempo_real = fin - inicio
    tiempo_simulado = tiempo_real / ESCALA_TIEMPO
    dias = tiempo_simulado / 24

    print(f"\nTiempo de simulación: {dias:.2f} días")

In [ ]:
N = 3     # Numero de camiones
X = 4     # Numero de viajes para ambas plantas

iniciar_simulacion(N, X)

Camión 1: CARGANDO EN BUENOS_AIRES
Camión 2: CARGANDO EN BUENOS_AIRES
Camión 3: CARGANDO EN BUENOS_AIRES
Camión 1: VIAJANDO
Camión 2: VIAJANDO
Camión 3: VIAJANDO
Camión 2: DESCARGANDO EN FERNANDEZ
Camión 1: DESCARGANDO EN FERNANDEZ
Camión 3: DESCARGANDO EN FERNANDEZ
Camión 2: CARGANDO EN FERNANDEZ
Camión 2: CARGANDO COMBUSTIBLE
Camión 1: CARGANDO EN FERNANDEZ
Camión 2: VIAJANDO
Camión 1: CARGANDO COMBUSTIBLE
Camión 3: CARGANDO EN FERNANDEZ
Camión 1: VIAJANDO
Camión 3: CARGANDO COMBUSTIBLE
Camión 3: VIAJANDO
Camión 2: DESCARGANDO EN BUENOS_AIRES
Camión 1: DESCARGANDO EN BUENOS_AIRES
Camión 2: CARGANDO EN BUENOS_AIRES
Camión 3: DESCARGANDO EN BUENOS_AIRES
Camión 2: VIAJANDO
Camión 2: DESCARGANDO EN FERNANDEZ
Camión 2: CARGANDO EN FERNANDEZ
Camión 2: CARGANDO COMBUSTIBLE
Camión 2: VIAJANDO
Camión 2: DESCARGANDO EN BUENOS_AIRES

Tiempo de simulación: 4.34 días
